# TP — Analyse de Sentiments

---

Un site de streaming reçoit chaque jour des milliers d'avis de spectateurs :
- *"Ce film est absolument magnifique, une masterpiece !"*
- *"Scénario nul, acteurs sans charisme, je veux récupérer mes 2h."*
- *"Bof, pas terrible mais pas non plus catastrophique."*

**Objectif** : entraîner un modèle qui lit un avis et prédit automatiquement s'il est **positif** ou **négatif**.

Nous travaillons sur le dataset **Allociné** — avis de films en français, étiquetés manuellement.

> **Prérequis** : TP Prétraitement (nettoyage, tokenisation, stopwords, lemmatisation, TF-IDF)

---
## Partie 1 — Théorie

---

### 1.1 — Qu'est-ce que l'analyse de sentiments ?

L'**analyse de sentiments** (ou *opinion mining*) est une tâche de NLP qui consiste à identifier l'**émotion** ou l'**opinion** exprimée dans un texte.

| Niveau | Exemple |
|---|---|
| **Binaire** | Positif / Négatif |
| **Ternaire** | Positif / Neutre / Négatif |
| **Échelle** | Score de 1 à 5 étoiles |
| **Aspects** | *"La caméra est bonne mais la batterie est nulle"* → caméra ✓, batterie ✗ |

Dans ce TP, on travaille au niveau **binaire** : Positif (1) ou Négatif (0).

#### Applications réelles
- Suivi de l'image de marque sur les réseaux sociaux
- Analyse des avis clients (Amazon, TripAdvisor…)
- Veille médiatique et politique
- Détection de haine en ligne

### 1.2 — Approche lexicale

L'approche **lexicale** s'appuie sur un **dictionnaire de sentiments** : une liste de mots avec leur polarité pré-calculée.

```
"magnifique" → +3.2     "catastrophique" → -2.8
"bon"        → +1.1     "nul"            → -2.0
"bof"        →  0.0     "décevant"       → -1.5
```

**Fonctionnement** :
1. Tokeniser le texte
2. Chercher chaque mot dans le lexique
3. Sommer les scores → score global positif ou négatif

#### Outils courants

| Outil | Langue | Notes |
|---|---|---|
| **VADER** | Anglais | Optimisé pour les réseaux sociaux, ponctuation, emojis |
| **TextBlob** | Anglais | Simple, rapide |
| **FEEL** | Français | Lexique de 14 000 entrées |
| **SentiWordNet** | Anglais | Basé sur WordNet |

> **Limite principale** : les dictionnaires ne comprennent pas le contexte. *"Ce film n'est pas bon"* → `bon` (+) donne un score positif alors que la phrase est négative !

### 1.3 — Approche supervisée

L'approche **supervisée** entraîne un modèle ML sur des avis déjà étiquetés. Le modèle apprend quels mots et combinaisons de mots sont associés à chaque sentiment.

```
Texte brut
    ↓ Prétraitement
Texte nettoyé + lemmatisé
    ↓ Vectorisation TF-IDF
Vecteur numérique
    ↓ Classifieur (LogReg / SVM / NB)
Sentiment prédit : Positif ou Négatif
```

**Avantages vs lexical** :
- Apprend le contexte (négation, ironie partielle)
- Adapté à la langue et au domaine du corpus d'entraînement
- Précision généralement bien supérieure

**Inconvénient** : nécessite des données étiquetées

### 1.4 — Régression Logistique

Dans ce TP, notre classifieur principal est la **Régression Logistique** — malgré son nom, c'est un classifieur, pas un outil de régression.

#### Principe

Elle modélise la **probabilité** qu'un texte appartienne à la classe Positif :

$$P(\text{Positif} \mid \mathbf{x}) = \frac{1}{1 + e^{-(\mathbf{w} \cdot \mathbf{x} + b)}}$$

où $\mathbf{x}$ est le vecteur TF-IDF, $\mathbf{w}$ les poids appris, $b$ le biais.

La **fonction sigmoïde** $\sigma(z) = \frac{1}{1+e^{-z}}$ écrase toute valeur réelle dans $]0, 1[$.

#### Hyperparamètre clé

| Paramètre | Rôle | Valeurs typiques |
|---|---|---|
| `C` | Inverse de la régularisation — C élevé = moins de pénalité = risque de surapprentissage | 0.01, 0.1, 1, 10 |

> **Pourquoi LogReg pour le texte ?** Elle est rapide, interprétable (les poids $\mathbf{w}$ montrent quels mots comptent) et très efficace sur des données TF-IDF creuses.

#### Avantages / Inconvénients

| Avantages | Inconvénients |
|---|---|
| Donne des **probabilités** (pas juste une classe) | Suppose une séparabilité approximativement linéaire |
| Très interprétable | Sensible aux features corrélées |
| Rapide à entraîner | Moins performante que SVM sur très grands vocabulaires |

### 1.5 — Comparaison des approches

| Critère | Lexicale (VADER) | Supervisée (LogReg / SVM) |
|---|---|---|
| **Données requises** | Aucune | Corpus étiqueté |
| **Précision** | Moyenne (~60-70 %) | Élevée (~85-95 %) |
| **Adaptée au français** | Non (VADER = anglais) | Oui (si corpus FR) |
| **Comprend la négation** | Partiellement | Mieux (via bigrammes) |
| **Interprétabilité** | Totale (poids du lexique) | Partielle (poids TF-IDF) |
| **Vitesse** | Très rapide | Entraînement requis |
| **Mise à jour** | Manuel | Réentraînement |

---
## Partie 2 — Pratique

---

### Étape 0 — Installation

In [ ]:
!pip install nltk scikit-learn vaderSentiment --quiet
!python -m spacy download fr_core_news_sm --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import nltk
nltk.download('punkt_tab',  quiet=True)
nltk.download('stopwords',  quiet=True)
nltk.download('wordnet',    quiet=True)
print('NLTK prêt.')

In [ ]:
import spacy
nlp = spacy.load('fr_core_news_sm')
print('SpaCy FR prêt.')

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
print('Sklearn prêt.')

---
### Étape 1 — Chargement du dataset

**Allociné** contient des avis de films en français avec leur sentiment :
- `0` → Négatif
- `1` → Positif

Le dataset a été bruité artificiellement pour simuler du texte réel (fautes, caractères spéciaux).

In [ ]:
df = pd.read_csv('../preprocessing/allocine_bruite.csv')
print(f'Taille : {df.shape}')
df.head(3)

In [ ]:
# Vérifier les colonnes et les valeurs
print('Colonnes :', df.columns.tolist())
print()
print('Distribution des sentiments :')
print(df['sentiment'].value_counts())

In [ ]:
# Visualisation de la distribution
counts = df['sentiment'].value_counts()
labels = ['Négatif (0)', 'Positif (1)']
colors = ['#e74c3c', '#2ecc71']

plt.figure(figsize=(5, 4))
plt.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
plt.title('Distribution des sentiments — Allociné', fontsize=13)
plt.ylabel('Nombre d'avis')
for i, v in enumerate(counts.values):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Quelques exemples par sentiment
print('=== Exemples d'avis ===\n')
for sentiment, label in [(1, 'POSITIF'), (0, 'NÉGATIF')]:
    print(f'--- {label} ---')
    exemples = df[df['sentiment'] == sentiment]['review'].sample(3, random_state=42)
    for ex in exemples:
        print(f'  • {ex[:120]}')
    print()

---
### Étape 2 — Prétraitement

On applique le même pipeline que dans le TP Prétraitement :
nettoyage → tokenisation → stopwords → lemmatisation → reconstruction.

#### 2.1 — Nettoyage regex

In [ ]:
def nettoyer(texte):
    texte = re.sub(r'[^a-zA-ZàâäéèêëîïôùûüçÀÂÄÉÈÊËÎÏÔÙÛÜÇ\s]', ' ', texte)
    texte = re.sub(r'\s+', ' ', texte)
    return texte.strip().lower()

print('BRUT    :', df['review'].iloc[0])
print('NETTOYÉ :', nettoyer(df['review'].iloc[0]))

In [ ]:
df['text_clean'] = df['review'].apply(nettoyer)
df[['review', 'text_clean']].head(3)

#### 2.2 — Tokenisation + Stopwords + Lemmatisation (SpaCy FR)

In [ ]:
from nltk.corpus import stopwords

stop_nltk  = set(stopwords.words('french'))
stop_spacy = nlp.Defaults.stop_words
stop_fr    = stop_nltk | stop_spacy
print(f'Stopwords FR (union) : {len(stop_fr)}')

In [ ]:
def pretraiter(texte):
    doc = nlp(texte)
    tokens = [
        tok.lemma_.lower()
        for tok in doc
        if tok.is_alpha and len(tok.lemma_) > 2 and tok.lemma_.lower() not in stop_fr
    ]
    return ' '.join(tokens)

# Démonstration
ex = df['text_clean'].iloc[0]
print('AVANT :', ex[:100])
print('APRÈS :', pretraiter(ex)[:100])

In [ ]:
print('Lemmatisation en cours (~3 min)...')
df['text_prep'] = df['text_clean'].apply(pretraiter)
print('Terminé.')
df[['review', 'text_prep']].head(3)

---
### Étape 3 — Approche lexicale : VADER

VADER (*Valence Aware Dictionary and sEntiment Reasoner*) est un lexique anglais optimisé pour les réseaux sociaux. On l'applique ici sur le français pour **illustrer ses limites**.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

# Démonstration sur quelques avis
print(f'{'Avis (50 premiers caractères)':<55} {'compound':>9} {'prédit':>8} {'réel':>6}')
print('-' * 82)
for _, row in df.sample(10, random_state=42).iterrows():
    scores  = vader.polarity_scores(row['review'])
    compound = scores['compound']
    predit  = 1 if compound >= 0 else 0
    print(f'{row["review"][:52]:<55} {compound:>9.3f} {predit:>8} {row["sentiment"]:>6}')

In [ ]:
# Application sur tout le dataset
df['score_vader'] = df['review'].apply(
    lambda t: vader.polarity_scores(t)['compound']
)
df['pred_vader'] = (df['score_vader'] >= 0).astype(int)

acc_vader = accuracy_score(df['sentiment'], df['pred_vader'])
print(f'Accuracy VADER sur Allociné (FR) : {acc_vader:.3f}')
print()
print(classification_report(df['sentiment'], df['pred_vader'],
                              target_names=['Négatif', 'Positif']))

> **Observation** : VADER obtient une accuracy proche du hasard (~50 %) sur du texte français — il ne comprend pas la langue. C'est la limite attendue de l'approche lexicale anglaise sur un corpus français.

---
### Étape 4 — Approche supervisée

#### 4.1 — Séparation Train / Test

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text_prep']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {len(X_train)} avis | Test : {len(X_test)} avis')
print(f'Distribution train : {y_train.value_counts().to_dict()}')

#### 4.2 — Vectorisation TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),    # unigrammes + bigrammes (ex: "pas bien", "très bon")
    sublinear_tf=True      # log(tf) — atténue les mots très fréquents
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec  = tfidf.transform(X_test)

print(f'Dimension : {X_train_vec.shape}')
print(f'({X_train_vec.shape[0]} avis × {X_train_vec.shape[1]} features)')

In [ ]:
# Top 10 features les plus discriminantes (score TF-IDF moyen)
import numpy as np
feature_names = tfidf.get_feature_names_out()
mean_scores   = np.asarray(X_train_vec.mean(axis=0)).flatten()
top10 = mean_scores.argsort()[::-1][:10]

pd.DataFrame({'mot / bigramme': feature_names[top10],
              'score moyen':    mean_scores[top10].round(4)})

#### 4.3 — Modèle 1 : Régression Logistique

In [ ]:
logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
logreg.fit(X_train_vec, y_train)
print('Entraînement terminé.')

In [ ]:
y_pred_lr = logreg.predict(X_test_vec)
acc_lr    = accuracy_score(y_test, y_pred_lr)
print(f'Accuracy Régression Logistique : {acc_lr:.3f}')
print()
print(classification_report(y_test, y_pred_lr, target_names=['Négatif', 'Positif']))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
ConfusionMatrixDisplay(cm_lr, display_labels=['Négatif', 'Positif']).plot(cmap='Blues')
plt.title('Matrice de confusion — Régression Logistique')
plt.tight_layout()
plt.show()

#### 4.4 — Modèle 2 : SVM (LinearSVC)

In [ ]:
svm = LinearSVC(C=1.0, random_state=42)
svm.fit(X_train_vec, y_train)

y_pred_svm = svm.predict(X_test_vec)
acc_svm    = accuracy_score(y_test, y_pred_svm)
print(f'Accuracy LinearSVC : {acc_svm:.3f}')
print()
print(classification_report(y_test, y_pred_svm, target_names=['Négatif', 'Positif']))

---
### Étape 5 — Comparaison des approches

In [ ]:
from sklearn.metrics import f1_score

resultats = pd.DataFrame({
    'Modèle':    ['VADER (lexical)', 'Régression Logistique', 'LinearSVC'],
    'Accuracy':  [acc_vader, acc_lr, acc_svm],
    'F1-score':  [
        f1_score(df['sentiment'], df['pred_vader'], average='weighted'),
        f1_score(y_test, y_pred_lr,  average='weighted'),
        f1_score(y_test, y_pred_svm, average='weighted'),
    ]
}).round(3)
resultats

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, ['Accuracy', 'F1-score']):
    colors = ['#e67e22', '#3498db', '#9b59b6']
    bars = ax.bar(resultats['Modèle'], resultats[metric], color=colors, edgecolor='white')
    ax.set_ylim(0, 1.05)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.3f}', ha='center', fontsize=9)

plt.suptitle('Comparaison : Lexical vs Supervisé — Allociné', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Mots les plus positifs et négatifs appris par LogReg
coefs        = logreg.coef_[0]
top_pos_idx  = coefs.argsort()[::-1][:15]
top_neg_idx  = coefs.argsort()[:15]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, idx, title, color in zip(
    axes,
    [top_pos_idx, top_neg_idx],
    ['Top 15 mots POSITIFS', 'Top 15 mots NÉGATIFS'],
    ['#2ecc71', '#e74c3c']
):
    mots   = feature_names[idx]
    scores = coefs[idx]
    ax.barh(mots[::-1], np.abs(scores[::-1]), color=color, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('|Poids|')

plt.suptitle('Mots les plus discriminants — Régression Logistique', fontsize=13)
plt.tight_layout()
plt.show()

---
### Étape 6 — Tester sur de nouveaux avis

On utilise le meilleur modèle supervisé pour prédire le sentiment de nouvelles phrases.

In [ ]:
meilleur_nom, meilleur_modele = (
    ('Régression Logistique', logreg) if acc_lr >= acc_svm
    else ('LinearSVC', svm)
)
print(f'Meilleur modèle : {meilleur_nom}')

In [ ]:
nouveaux_avis = [
    "Un film magnifique, une histoire touchante et des acteurs brillants.",
    "Complètement nul, scénario vide, acteurs sans charisme. À éviter.",
    "Pas mal, quelques bons moments mais globalement décevant.",
    "Chef-d'œuvre absolu ! Je recommande à tout le monde.",
    "Pire film que j'ai vu depuis des années, un vrai gâchis.",
]

avis_prep  = [pretraiter(nettoyer(a)) for a in nouveaux_avis]
avis_vec   = tfidf.transform(avis_prep)
predictions = meilleur_modele.predict(avis_vec)

print(f'Modèle : {meilleur_nom}\n')
for avis, pred in zip(nouveaux_avis, predictions):
    emoji = '✅ Positif' if pred == 1 else '❌ Négatif'
    print(f'{emoji} | {avis}')

---
### Bilan

| Approche | Accuracy | Avantage | Limite |
|---|---|---|---|
| **VADER** (lexical) | ~50 % | Aucune donnée requise | Anglais uniquement |
| **Régression Logistique** | ~87 % | Probabilités + interprétable | Données étiquetées |
| **LinearSVC** | ~88 % | Très performant sur texte | Moins interprétable |

**Conclusion** : pour du texte français, l'approche supervisée est indispensable. Les bigrammes (`pas bien`, `très décevant`) permettent de capturer partiellement la négation — un avantage clé sur l'approche lexicale naïve.